In [ ]:
__call__ 是 Python 的一种特殊方法，也叫“魔术方法”。它的作用是：让对象可以像函数一样，加括号调用。
例如：
class Adder:
    def __call__(self, a, b):
        return a + b

obj = Adder()     # 创建对象
result = obj(2, 3)  # Python 自动调用 __call__
print(result)      # 5
当你写：
obj(2, 3)
可以理解为 Python 自动执行了：
obj.__call__(2, 3)
这里的 self 指的是 obj，由 Python 自动传入；a 和 b 分别接收 2 和 3。
注意，方法名必须是 __call__，前后各两个下划线。这是 Python 约定好的名字，写成普通的 call 就没有这个自动触发的效果。
把它和你刚看到的 __init__ 放在一起：
特殊方法	什么时候自动执行	用途
__init__	创建对象、进行初始化时	设置对象的初始状态
__call__	对象后面加括号调用时	执行你定义的操作


对应你的代码：
criterion = nn.CrossEntropyLoss()  # 创建对象，初始化时执行 __init__
loss = criterion(logits, y)        # 调用对象，触发继承来的 __call__ 机制

用一个简化例子就能看清楚：
class Parent:
    def __call__(self, x):
        return self.forward(x)

class Child(Parent):
    def forward(self, x):
        return x * 2

obj = Child()
print(obj(3))  # 6
Child 没写 __call__，但继承了父类的 __call__，所以 obj(3) 仍然能执行。 父类里的 self 就是这个 obj，因此 self.forward(x) 会调用 Child 定义的 forward。
另外，注意区分这两处：
criterion = nn.CrossEntropyLoss()  # 创建对象，执行初始化 __init__
loss = criterion(logits, y)        # 调用对象，走 __call__ → forward
这里的 __call__ 是解释第二行为什么能这样写的。

# nn.CrossEntropyLoss() 这里为什么要加括号

因为 nn.CrossEntropyLoss 是一个类，加上 () 才会创建这个类的对象。
criterion = nn.CrossEntropyLoss    # 不加括号：criterion 指向这个类
criterion = nn.CrossEntropyLoss()  # 加上括号：创建对象，赋值给 criterion
可以把“类”理解成设计图，“对象”是按照设计图造出来的东西。
所以你的代码分成两步：
# 第一步：创建损失计算器
criterion = nn.CrossEntropyLoss()

# 第二步：用这个计算器计算损失
loss = criterion(logits, y)

# 为什么是 criterion(logits, y)？
计算误差需要两样东西：
logits = model(x)             # 模型的预测：每个类别的原始得分
loss = criterion(logits, y)   # y 是正确答案
例如，要把图片分成“猫、狗、鸟”三类，一个样本的输出可能是：
logits = torch.tensor([[2.0, 1.0, 0.1]])  # 猫、狗、鸟的原始得分
y = torch.tensor([0])                     # 正确类别是猫，编号为 0
对于这种用类别编号表示的标签，交叉熵的计算等价于：
先把得分转成概率 → 取出正确类别的概率 → 计算它的负对数。
原始得分：       [2.0,   1.0,   0.1]
Softmax 概率：  [0.659, 0.242, 0.099]
正确类别：      0，也就是猫
损失：          -ln(0.659) ≈ 0.417
公式就是：
\[
L = -\log p_{\text{正确类别}}
\]因此：
模型给正确类别的概率	损失
0.9	约 0.105
0.5	约 0.693
0.1	约 2.303


使用 CrossEntropyLoss 时直接传原始 logits，不用自己先做 Softmax。 它内部以数值更稳定的方式完成等价计算。

In [ ]:
import torch.nn as nn
criterion = nn.CrossEntropyLoss()

class mlp(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(28*28, 256)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x

model = mlp()
